# Phase 3b: CascadeGPS Classifier

From-scratch GraphGPS variant tailored to rumor cascades — see `models/cascade_gps.py`.

Key differences from `GPSClassifier` (notebook 03):
- Edge-aware attention: `edge_attr` (Δt + direction flag) is projected into per-head bias added to Q·Kᵀ.
- Pairwise |Δt| attention bias across the dense batch.
- Per-node gated fusion of MPNN + attention branches.
- Direction-gated GINE-style MPNN (multiplicative edge gate).
- Sinusoidal time encoding on the raw timestamp.
- Learnable ROOT bias added to graph-local node 0.

Runs on Twitter15 and Twitter16 with the same 5 seeds, split, and trainer config as the stock GPS notebook for an apples-to-apples comparison.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import os
import torch
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

os.makedirs("../results", exist_ok=True)

from data.dataset import CascadeDataset
from models.cascade_gps import CascadeGPSClassifier
from training.trainer import run_experiment

In [ ]:
DATA_ROOT = "../Twitter15_16_dataset-main"
SEEDS = [0, 1, 2, 3, 4]

DEVICE0 = "cuda:0" if torch.cuda.is_available() else "cpu"
DEVICE1 = "cuda:1" if torch.cuda.device_count() > 1 else DEVICE0
DEVICE = DEVICE0  # kept for any single-GPU cells
print(f"GPU 0: {DEVICE0}  |  GPU 1: {DEVICE1}")

In [ ]:
ds15 = CascadeDataset(root=DATA_ROOT, name="twitter15")
ds16 = CascadeDataset(root=DATA_ROOT, name="twitter16")
print(f"Twitter15: {len(ds15)} graphs, x={ds15[0].x.shape}, edge_attr={ds15[0].edge_attr.shape}")
print(f"Twitter16: {len(ds16)} graphs, x={ds16[0].x.shape}, edge_attr={ds16[0].edge_attr.shape}")

IN_CHANNELS = ds15[0].x.shape[1]   # 30
EDGE_DIM    = ds15[0].edge_attr.shape[1]  # 2
print(f"in_channels={IN_CHANNELS}, edge_dim={EDGE_DIM}")

In [ ]:
CASCADE_GPS_KWARGS = dict(
    in_channels=IN_CHANNELS,
    hidden_channels=128,
    num_layers=3,
    heads=4,
    dropout=0.1,
    edge_dim=EDGE_DIM,
)

In [ ]:
# ── RESUME CELL ─────────────────────────────────────────────────────────────
# Seed 3 → cuda:0, seed 4 → cuda:1, running in parallel.

import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

COMPLETED_SEEDS_T15 = [
    {"seed": 0, "val_f1": 0.4397, "test_acc": 0.4329, "test_f1": 0.4239},
    {"seed": 1, "val_f1": 0.4627, "test_acc": 0.4195, "test_f1": 0.4102},
    {"seed": 2, "val_f1": 0.4271, "test_acc": 0.4329, "test_f1": 0.4096},
]

RESUME_SEEDS = [3, 4]
RESUME_DEVICES = [DEVICE0, DEVICE1]

print(f"Resuming Twitter15: seed 3 → {DEVICE0}, seed 4 → {DEVICE1}")

def run_one_seed_t15(seed, device):
    return run_experiment(
        CascadeGPSClassifier, CASCADE_GPS_KWARGS, ds15, [seed],
        epochs=200, lr=1e-3, weight_decay=0.05, patience=60,
        warmup_ratio=0.1, lap_pe_sign_flip=True,
        max_nodes_per_batch=2048, device=device,
    )

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {pool.submit(run_one_seed_t15, s, d): s
               for s, d in zip(RESUME_SEEDS, RESUME_DEVICES)}
    resume_results = {}
    for f in as_completed(futures):
        s = futures[f]
        resume_results[s] = f.result()
        print(f"  seed={s} done")

resume_per_seed = []
for s in sorted(resume_results):
    resume_per_seed.extend(resume_results[s]["per_seed"])

all_seed_results = COMPLETED_SEEDS_T15 + resume_per_seed
test_f1s  = [r["test_f1"]  for r in all_seed_results]
test_accs = [r["test_acc"] for r in all_seed_results]
res15 = {
    "per_seed":      all_seed_results,
    "test_f1_mean":  float(np.mean(test_f1s)),
    "test_f1_std":   float(np.std(test_f1s)),
    "test_acc_mean": float(np.mean(test_accs)),
    "test_acc_std":  float(np.std(test_accs)),
}

print(f"\nTwitter15 CascadeGPS (5 seeds) — "
      f"Acc: {res15['test_acc_mean']:.3f} ± {res15['test_acc_std']:.3f}  "
      f"F1: {res15['test_f1_mean']:.3f} ± {res15['test_f1_std']:.3f}")

with open("../results/cascade_gps_twitter15.json", "w") as f:
    json.dump(res15, f, indent=2)
print("Saved → results/cascade_gps_twitter15.json")

In [ ]:
print("=" * 50)
print("CascadeGPS — Twitter15  (seeds 0–4, full run)")
print("=" * 50)
res15 = run_experiment(
    CascadeGPSClassifier, CASCADE_GPS_KWARGS, ds15, SEEDS,
    epochs=200,
    lr=1e-3,
    weight_decay=0.05,
    patience=60,
    warmup_ratio=0.1,
    lap_pe_sign_flip=True,
    max_nodes_per_batch=2048,
    device=DEVICE,
)
print(f"\nTwitter15 CascadeGPS — Acc: {res15['test_acc_mean']:.3f} ± {res15['test_acc_std']:.3f}  "
      f"F1: {res15['test_f1_mean']:.3f} ± {res15['test_f1_std']:.3f}")

with open("../results/cascade_gps_twitter15.json", "w") as f:
    json.dump(res15, f, indent=2)

In [ ]:
print("=" * 50)
print("CascadeGPS — Twitter16  (seeds 0-2 → cuda:0, seeds 3-4 → cuda:1)")
print("=" * 50)

from concurrent.futures import ThreadPoolExecutor, as_completed

T16_GROUPS = [
    ([0, 1, 2], DEVICE0),
    ([3, 4],    DEVICE1),
]

def run_t16_group(seeds, device):
    return run_experiment(
        CascadeGPSClassifier, CASCADE_GPS_KWARGS, ds16, seeds,
        epochs=200, lr=1e-3, weight_decay=0.05, patience=60,
        warmup_ratio=0.1, lap_pe_sign_flip=True,
        max_nodes_per_batch=2048, device=device,
    )

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {pool.submit(run_t16_group, seeds, device): seeds
               for seeds, device in T16_GROUPS}
    group_results = []
    for f in as_completed(futures):
        seeds = futures[f]
        r = f.result()
        group_results.append(r)
        print(f"  seeds={seeds} done")

all_per_seed = []
for r in group_results:
    all_per_seed.extend(r["per_seed"])
all_per_seed.sort(key=lambda x: x["seed"])

import numpy as np
test_f1s  = [r["test_f1"]  for r in all_per_seed]
test_accs = [r["test_acc"] for r in all_per_seed]
res16 = {
    "per_seed":      all_per_seed,
    "test_f1_mean":  float(np.mean(test_f1s)),
    "test_f1_std":   float(np.std(test_f1s)),
    "test_acc_mean": float(np.mean(test_accs)),
    "test_acc_std":  float(np.std(test_accs)),
}

print(f"\nTwitter16 CascadeGPS — Acc: {res16['test_acc_mean']:.3f} ± {res16['test_acc_std']:.3f}  "
      f"F1: {res16['test_f1_mean']:.3f} ± {res16['test_f1_std']:.3f}")

with open("../results/cascade_gps_twitter16.json", "w") as f:
    json.dump(res16, f, indent=2)

In [ ]:
# Comparison table — load all model results from saved JSON
def load_result(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

gcn15        = load_result("../results/gcn_twitter15.json")
gat15        = load_result("../results/gat_twitter15.json")
bigcn15      = load_result("../results/bigcn_twitter15.json")
gps15        = load_result("../results/gps_twitter15.json")
gcn16        = load_result("../results/gcn_twitter16.json")
gat16        = load_result("../results/gat_twitter16.json")
bigcn16      = load_result("../results/bigcn_twitter16.json")
gps16        = load_result("../results/gps_twitter16.json")

rows = [
    ("GCN",        "Twitter15", gcn15),
    ("GAT",        "Twitter15", gat15),
    ("BiGCN",      "Twitter15", bigcn15),
    ("GPS",        "Twitter15", gps15),
    ("CascadeGPS", "Twitter15", res15),
    ("GCN",        "Twitter16", gcn16),
    ("GAT",        "Twitter16", gat16),
    ("BiGCN",      "Twitter16", bigcn16),
    ("GPS",        "Twitter16", gps16),
    ("CascadeGPS", "Twitter16", res16),
]

print()
print("=" * 70)
print(f"{'Model':<12} {'Dataset':<14} {'Acc mean±std':>16}  {'Macro-F1 mean±std':>18}")
print("-" * 70)
for model, ds_name, res in rows:
    if res is None:
        print(f"{model:<12} {ds_name:<14}  (no results yet)")
        continue
    a_m, a_s = res['test_acc_mean'], res['test_acc_std']
    f_m, f_s = res['test_f1_mean'], res['test_f1_std']
    print(f"{model:<12} {ds_name:<14} {a_m:.3f} ± {a_s:.3f}        {f_m:.3f} ± {f_s:.3f}")
print("=" * 70)
print("(5 seeds, 60/20/20 stratified split, best val-F1 checkpoint)")